# Customer Churn Prediction — End-to-End Machine Learning Project

**Dataset:** IBM Telco Customer Churn sample dataset  
**Objective:** Predict whether a telecom customer is likely to leave the service (`Churn = Yes`).

> Important: Run the notebook from top to bottom. All dataset counts and model metrics must come from your actual run. Do not invent or copy unverified metrics into a resume.

## Project workflow
1. Load and understand the dataset
2. Clean data and inspect missing values / duplicates
3. Explore churn patterns with visualizations
4. Prepare features and target
5. Split data into training and test sets
6. Preprocess numeric and categorical features using a leakage-safe pipeline
7. Compare models using cross-validation on training data
8. Evaluate the selected model once on the held-out test set
9. Save the fitted pipeline and evaluation results for the Streamlit app

## 1. Install and import libraries

If a package is missing, run the installation command in a notebook cell.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme()
RANDOM_STATE = 42

## 2. Load the dataset

The notebook first checks for `Telco-Customer-Churn.csv` in the project folder. If it is not found, it downloads the public IBM sample CSV. Internet access is required for the download option.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/IBM/"
    "telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
)
DATA_PATH = Path("Telco-Customer-Churn.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded local dataset: {DATA_PATH.resolve()}")
else:
    df = pd.read_csv(DATA_URL)
    print("Loaded dataset from IBM's public GitHub repository.")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand the dataset

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()

In [ ]:
display(df.describe(include="all").T)

## 4. Data quality checks: missing values and duplicates

`TotalCharges` may contain blank strings. Convert it to numeric so blank entries become missing values (`NaN`), then inspect them. Do not remove rows automatically without understanding their impact.

In [ ]:
print("Missing values before TotalCharges conversion:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("\nMissing values after conversion:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nDuplicate full rows:", df.duplicated().sum())
if "customerID" in df.columns:
    print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

In [ ]:
# Remove exact duplicate rows only. Keep the first occurrence.
df = df.drop_duplicates().copy()

# Check target labels and missing target values.
if "Churn" not in df.columns:
    raise ValueError("Target column 'Churn' was not found.")

print("Target value counts:")
display(df["Churn"].value_counts(dropna=False).to_frame("count"))

## 5. Exploratory Data Analysis (EDA)

These visualizations help identify class imbalance and relationships between customer attributes and churn. They describe associations; they do not prove causation.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn", order=["No", "Yes"])
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

print("Churn percentage:")
display((df["Churn"].value_counts(normalize=True) * 100).round(2).to_frame("percentage"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x="tenure", hue="Churn", bins=25, kde=True, ax=axes[0])
axes[0].set_title("Tenure Distribution by Churn")
sns.histplot(data=df, x="MonthlyCharges", hue="Churn", bins=25, kde=True, ax=axes[1])
axes[1].set_title("Monthly Charges by Churn")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Contract Type vs Customer Churn")
plt.xlabel("Contract type")
plt.ylabel("Number of customers")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

In [ ]:
numeric_for_eda = df.select_dtypes(include="number").columns.tolist()
if numeric_for_eda:
    plt.figure(figsize=(8, 5))
    sns.heatmap(df[numeric_for_eda].corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap — Numeric Features")
    plt.tight_layout()
    plt.show()

In [ ]:
numeric_for_boxplot = [c for c in ["tenure", "MonthlyCharges", "TotalCharges"] if c in df.columns]
for col in numeric_for_boxplot:
    plt.figure(figsize=(7, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()

## 6. Outlier analysis

The IQR method below flags potential outliers in numeric columns. In customer datasets, extreme values may be genuine (for example, long tenure or high total charges). Review them rather than deleting them automatically. The model pipeline uses median imputation and scaling; no rows are removed solely because they are outliers.

In [ ]:
def iqr_outlier_summary(data, columns):
    rows = []
    for col in columns:
        series = pd.to_numeric(data[col], errors="coerce").dropna()
        if series.empty:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = ((series < lower) | (series > upper)).sum()
        rows.append({
            "column": col, "Q1": q1, "Q3": q3, "IQR": iqr,
            "lower_bound": lower, "upper_bound": upper,
            "potential_outliers": int(count)
        })
    return pd.DataFrame(rows)

numeric_columns_eda = df.select_dtypes(include="number").columns.tolist()
display(iqr_outlier_summary(df, numeric_columns_eda))

## 7. Prepare features and target

`Churn` is the target: `Yes` is mapped to 1 and `No` to 0. `customerID` is removed because it is an identifier rather than a customer-behavior feature. Unknown target labels are excluded.

In [ ]:
target_col = "Churn"
required_columns = {
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges", "Churn"
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {sorted(missing_columns)}")

df[target_col] = df[target_col].astype(str).str.strip()
df = df[df[target_col].isin(["Yes", "No"])].copy()

y = df[target_col].map({"No": 0, "Yes": 1}).astype(int)
X = df.drop(columns=[target_col, "customerID"])

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Modeling records:", len(X))
print("Number of numeric features:", len(numeric_features), numeric_features)
print("Number of categorical features:", len(categorical_features), categorical_features)
print("\nTarget distribution:")
display(y.value_counts().rename(index={0: "No churn", 1: "Churn"}).to_frame("count"))

## 8. Split into training and test sets

Use an 80:20 stratified split. The test set is held aside and should only be used for the final evaluation after model selection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 9. Build a leakage-safe preprocessing pipeline

- Numeric features: median imputation followed by standardization.
- Categorical features: most-frequent imputation followed by one-hot encoding.
- These transformations are fitted inside each model pipeline using training folds only.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 10. Train and compare models using cross-validation

Model selection is based on mean F1-score across stratified folds of the training set, not on the final test set. F1 is useful when the churn and non-churn classes are imbalanced. The baseline provides a simple reference point.

In [ ]:
models = {
    "Baseline (most frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_rows = []
for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=-1, error_score="raise"
    )
    cv_rows.append({
        "Model": model_name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV F1", ascending=False).reset_index(drop=True)
display(cv_results.round(4))

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=cv_results, x="CV F1", y="Model")
plt.title("Cross-Validation Model Comparison (F1-score)")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## 11. Select the best model and evaluate once on the test set

The candidate with the highest mean cross-validation F1-score is selected. It is then fitted on all training data and evaluated on the untouched test set.

In [ ]:
best_model_name = cv_results.iloc[0]["Model"]
best_estimator = models[best_model_name]

best_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_estimator)
])
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_probability = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "Selected model": best_model_name,
    "Test accuracy": accuracy_score(y_test, y_pred),
    "Test precision": precision_score(y_test, y_pred, zero_division=0),
    "Test recall": recall_score(y_test, y_pred, zero_division=0),
    "Test F1-score": f1_score(y_test, y_pred, zero_division=0),
    "Test ROC-AUC": roc_auc_score(y_test, y_probability)
}
display(pd.DataFrame([test_metrics]).T.rename(columns={0: "value"}))
print("\nClassification report:")
print(classification_report(
    y_test, y_pred, target_names=["No churn", "Churn"], zero_division=0
))

## 12. Confusion matrix and ROC curve

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["No churn", "Churn"], cmap="Blues"
)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title("ROC Curve — Test Set")
plt.tight_layout()
plt.show()

## 13. Save the trained pipeline and results

The saved pipeline includes preprocessing and the selected model. The Streamlit app loads this pipeline so it applies the same transformations to new customer inputs.

In [ ]:
OUTPUT_DIR = Path(".")
joblib.dump(best_model, OUTPUT_DIR / "churn_model.joblib")
cv_results.to_csv(OUTPUT_DIR / "model_comparison_cv.csv", index=False)

with open(OUTPUT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved: churn_model.joblib")
print("Saved: model_comparison_cv.csv")
print("Saved: test_metrics.json")

## 14. Final project conclusion

After running the notebook, complete this conclusion using your actual output:

- The dataset contained **[actual row count]** records and **[actual column count]** columns before feature preparation.
- The analysis identified customer churn patterns related to tenure, contract type, monthly charges and other service attributes. Mention only patterns supported by your charts.
- The selected model was **[model name]**, chosen using training-set cross-validation.
- On the held-out test set, it achieved **[actual accuracy] accuracy**, **[actual precision] precision**, **[actual recall] recall**, **[actual F1] F1-score**, and **[actual ROC-AUC] ROC-AUC**.
- The model and preprocessing pipeline were saved for use in the Streamlit application.

**Limitations:** This is a sample educational dataset. Predictions are estimates, not guarantees. Model performance may change on new customers or different telecom datasets.

## 15. Checklist before publishing

- [ ] Run every notebook cell from top to bottom without errors.
- [ ] Confirm actual dataset shape and data-quality counts.
- [ ] Review EDA charts and write findings based on evidence.
- [ ] Record cross-validation and final test metrics.
- [ ] Confirm `churn_model.joblib` is created.
- [ ] Run the Streamlit app.
- [ ] Do not put placeholder metrics on your resume.